# Complexity-08b — Origami Wolfram pli 4 : les combinateurs S/K/I, ou la representation ultime du calcul

**Navigation** : [<< Complexity-08 — La règle 30 comme générateur d'aléa, la règle 110 et son universalité](Complexity-08-WolframRule30Aleatoire-Rule110Universelle-Python.ipynb) | [Index de la série](README.md) | [Complexity-08c — Machines de Gödel →](Complexity-08c-MachinesDeGoedel-Python.ipynb) | [jumeau Python](Complexity-08b-CombinateursSKI-TagDePost-Python.ipynb)

**Carnet** : deux points de la definition du pli 4 qu'aucun carnet de la serie ne couvre encore --
la **logique combinatoire S/K/I** (syntaxe, reduction, forme normale, abstraction de bracket,
entiers de Church) et le **probleme de tag de Post** comme borne d'irreductibilite du geste
symbolique, avec, a cote de chaque these, sa **reception critique**.

**Organe utilisé** : le jumeau Python importe le module `ict.combinators` sous le nom `cb`.
Cette version porte ses fonctions de réduction S/K/I et d'évolution du tag de Post dans la
classe C# `Combinateurs`, avec des types et des algorithmes .NET natifs. Les instruments de
présentation du carnet (tables de traces, comparaisons de tailles) restent distincts du moteur.

**Ce que ce carnet fait, et ce qu'il ne fait pas.**

| Ces affirmations | Statut | Ce que fait ce carnet |
|---|---|---|
| S et K suffisent a representer tout calcul | *theoreme* (Schonfinkel 1924 ; Church 1936) | il **demonstre le mecanisme sur des exemples** (derivation de I, entiers de Church, arithmetique) ; il ne re-prouve pas l'universalite |
| La reduction est confluente : toute forme normale est unique (Church-Rosser) | *theoreme* | il en donne le **temoin empirique** (ordres de reduction aleatoires -> une seule forme normale) ; la preuve formelle est un grain separe de l'EPIC (organe Lean) |
| Le systeme de tag de Post (0->00, 1->1101) s'arrete-t-il pour toute graine ? | *probleme ouvert* depuis 1921 | il **mesure** les comportements (arret, cycle, croissance) et les blocs inatteignables ; il ne tranche pas |

La distinction est le sujet d'une partie de ce carnet : un theoreme demontre sur exemples, un
temoin empirique et un probleme ouvert ne se citent pas de la meme facon.

> **Statut épistémique** — **Sans verdict à ce jour** : ce carnet n'a pas encore de ligne dans la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md), donc aucune solidité n'est revendiquée ici. Ce qu'il mesure est reproductible depuis ses cellules ; ce n'est pas un acquis de la série.

## Sources primaires et regle de citation

Les textes utilises sont lus dans les copies archivees du gisement
`G:\Mon Drive\MyIA\IA\Bibliographie IA\` :

| Reference | Copie archivee |
|---|---|
| Wolfram, *Combinators: A Centennial View* (2020) | `Symbolic\Wolfram on Mathematics and Proof\2020 - Wolfram - Combinators - A Centennial View.html` |
| Wolfram, *After 100 Years, Can We Finally Crack Post's Problem of Tag?* (4 mars 2021) | `Automata\Wolfram Physics Project\2021 - Wolfram - After 100 Years, Can We Finally Crack Post's Problem of Tag.html` |
| Wolfram, *1920, 2020 and a $20,000 Prize: Announcing the S Combinator Challenge* (8 juin 2021) | `Symbolic\Wolfram on Mathematics and Proof\2021 - Wolfram - 1920-2020 and a $20000 Prize, Announcing the S Combinator Challenge.html` |
| Wolfram, *A Bibliography of Combinators* (arXiv 2106.11729, 2021) | `Symbolic\Wolfram on Mathematics and Proof\2021 - Wolfram - A Bibliography of Combinators (arXiv 2106.11729).pdf` |

**Convention de citation** : chaque affirmation attribuee a un texte cite la **section de la copie
archivee** (une capture HTML se cite par son titre de section, jamais par un numero de ligne ; le
PDF bibliographique par sa page). La reception critique (Curry, Hindley, Cardone, Bimbo, Klop)
est relevee dans la bibliographie archivee ci-dessus, pas dans les originaux -- et le carnet le
dit quand c'est le cas.

In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Port manuel natif de l'organe Python `ict.combinators`.
// Le moteur S/K/I et le systeme de tag de Post sont ecruits ici en types et
// algorithmes .NET natifs ; les instruments de presentation du carnet restent
// distincts du moteur.

// Les affichages numeriques sont invariants : point decimal, jamais de virgule.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

Console.WriteLine("Combinateurs (S/K/I + tag de Post) : port manuel natif en C#");
Console.WriteLine("Regles : S f g x = f x (g x) | K x y = x | I x = x");

// Termes S/K/I : arbre binaire d'application, association a gauche.
public abstract class Term
{
    public static readonly Term S = new Var("S");
    public static readonly Term K = new Var("K");
    public static readonly Term I = new Var("I");

    // Le terme omega : S I I (S I I) se reduit en lui-meme -- divergence pure.
    public static readonly Term Omega = new App(new App(new App(S, I), I), new App(new App(S, I), I));
}

// Variable libre (affichage seulement : les combinateurs clos n'en ont pas).
public sealed class Var : Term
{
    public string Name { get; }

    public Var(string name) => Name = name;

    public override string ToString() => Name;
}

// Application : Func applique a Arg (association a gauche).
public sealed class App : Term
{
    public Term Func { get; }
    public Term Arg { get; }

    public App(Term func, Term arg)
    {
        Func = func;
        Arg = arg;
    }

    public override string ToString() => Combinateurs.ToStr(this);
}

// Moteur combinatoire : syntaxe, reduction, bracket, entiers de Church, tag de Post.
public static class Combinateurs
{
    // Composition, decomposee en S/K : B f g x = f (g x).
    public static readonly Term B = new App(new App(Term.S, new App(Term.K, Term.S)), Term.K);

    // Regles du systeme de tag de Post (1921) : 0 -> 00, 1 -> 1101, nu = 3.
    public static readonly IReadOnlyDictionary<char, string> PostRules = new Dictionary<char, string> { ['0'] = "00", ['1'] = "1101" };

    public const int PostNu = 3;

    // --- Syntaxe -------------------------------------------------------------

    // Parse une chaine d'applications "S K K x" (association a gauche).
    // Jetons reconnus : S, K, I, B (combinateur usuel decompose en S/K) ;
    // tout autre identifiant devient une variable libre.
    public static Term ParseSki(string text)
    {
        string[] tokens = text.Replace("(", " ( ").Replace(")", " ) ").Split(' ', StringSplitOptions.RemoveEmptyEntries);
        int pos = 0;

        Term Build(List<Term> atoms)
        {
            Term t = atoms[0];
            for (int i = 1; i < atoms.Count; i++)
                t = new App(t, atoms[i]);
            return t;
        }

        Term ParseGroup()
        {
            var atoms = new List<Term>();
            while (pos < tokens.Length)
            {
                string tok = tokens[pos++];
                if (tok == ")")
                    break;
                if (tok == "(")
                {
                    atoms.Add(ParseGroup());
                    continue;
                }

                atoms.Add(Atom(tok));
            }

            if (atoms.Count == 0)
                throw new ArgumentException("terme vide");
            return Build(atoms);
        }

        var top = new List<Term>();
        while (pos < tokens.Length)
        {
            string tok = tokens[pos++];
            if (tok == ")")
                throw new ArgumentException("parenthese fermante inattendue");
            if (tok == "(")
            {
                top.Add(ParseGroup());
                continue;
            }

            top.Add(Atom(tok));
        }

        if (top.Count == 0)
            throw new ArgumentException("terme vide");
        return Build(top);
    }

    // Mappe un jeton a son terme ; B est decompose en S/K.
    public static Term Atom(string tok)
    {
        if (tok == "S")
            return Term.S;
        if (tok == "K")
            return Term.K;
        if (tok == "I")
            return Term.I;
        if (tok == "B")
            return B;
        return new Var(tok);
    }

    // Affichage parenthese minimal : seul l'argument de droite prend des
    // parentheses quand il est lui-meme une application.
    public static string ToStr(Term term)
    {
        if (term is Var v)
            return v.Name;
        var a = (App)term;
        string left = ToStr(a.Func);
        string right = ToStr(a.Arg);
        if (a.Arg is App)
            right = "(" + right + ")";
        return left + " " + right;
    }

    // --- Reduction -----------------------------------------------------------

    static bool MatchS(Term t, out Term f, out Term g, out Term x)
    {
        f = Term.I;
        g = Term.I;
        x = Term.I;
        if (t is App a && a.Func is App a2 && a2.Func is App a3 && ReferenceEquals(a3.Func, Term.S))
        {
            f = a3.Arg;
            g = a2.Arg;
            x = a.Arg;
            return true;
        }

        return false;
    }

    static bool MatchK(Term t, out Term x, out Term y)
    {
        x = Term.I;
        y = Term.I;
        if (t is App a && a.Func is App a2 && ReferenceEquals(a2.Func, Term.K))
        {
            x = a2.Arg;
            y = a.Arg;
            return true;
        }

        return false;
    }

    static bool MatchI(Term t, out Term x)
    {
        x = Term.I;
        if (t is App a && ReferenceEquals(a.Func, Term.I))
        {
            x = a.Arg;
            return true;
        }

        return false;
    }

    // Un pas au redex racine ; null si la racine n'est pas un redex.
    public static Term? ApplyRedex(Term term)
    {
        if (MatchS(term, out Term f, out Term g, out Term x))
            return new App(new App(f, x), new App(g, x));
        if (MatchK(term, out Term kx, out Term ky))
            return kx;
        if (MatchI(term, out Term ix))
            return ix;
        return null;
    }

    // Vrai si aucun sous-terme n'est un redex S/K/I.
    public static bool IsNormalForm(Term term)
    {
        if (ApplyRedex(term) != null)
            return false;
        if (term is App a)
            return IsNormalForm(a.Func) && IsNormalForm(a.Arg);
        return true;
    }

    // Un pas de reduction.
    // order = "normal"     : redex le plus a gauche, l'exterieur d'abord
    //                       (trouve la forme normale quand elle existe).
    // order = "applicative" : l'interieur d'abord (evalue les arguments ;
    //                       peut diverger sur un terme qui a une forme normale).
    public static Term? ReduceOnce(Term term, string order = "normal")
    {
        if (order != "normal" && order != "applicative")
            throw new ArgumentException("order doit valoir 'normal' ou 'applicative'");
        return Step(term, order == "applicative");
    }

    static Term? Step(Term t, bool innermost)
    {
        if (!innermost)
        {
            Term? r = ApplyRedex(t);
            if (r != null)
                return r;
        }

        if (t is App a)
        {
            Term? f = Step(a.Func, innermost);
            if (f != null)
                return new App(f, a.Arg);
            Term? arg = Step(a.Arg, innermost);
            if (arg != null)
                return new App(a.Func, arg);
        }

        if (innermost)
            return ApplyRedex(t);
        return null;
    }

    // Reduit jusqu'a la forme normale ou maxSteps.
    // Renvoie (forme normale ou null = divergence, nombre de pas, trace optionnelle).
    public static (Term? FormeNormale, int Pas, List<Term>? Trace) ReduceTerm(Term term, string order = "normal", int maxSteps = 10000, bool trace = false)
    {
        bool innermost = order == "applicative";
        int steps = 0;
        List<Term>? tr = trace ? new List<Term> { term } : null;
        Term current = term;
        while (steps < maxSteps)
        {
            Term? nxt = Step(current, innermost);
            if (nxt == null)
                return (current, steps, tr);
            current = nxt;
            steps++;
            tr?.Add(current);
        }

        return (null, steps, tr);
    }

    // Reduction en ordre aleatoire (n'importe quel redex, uniformement).
    // Instrument du constat empirique de Church-Rosser : des ordres differents
    // convergent vers la meme forme normale (unicite).
    public static (Term? FormeNormale, int Pas) RandomReduce(Term term, Random rng, int maxSteps = 10000)
    {
        int steps = 0;
        Term current = term;
        while (steps < maxSteps)
        {
            List<List<int>> paths = AllRedexPaths(current);
            if (paths.Count == 0)
                return (current, steps);
            List<int> chosen = paths[rng.Next(paths.Count)];
            current = ReplacePath(current, chosen);
            steps++;
        }

        return (null, steps);
    }

    // Chemins (0 = func, 1 = arg) de tous les redex du terme.
    static List<List<int>> AllRedexPaths(Term term)
    {
        var res = new List<List<int>>();
        void Walk(Term t, List<int> path)
        {
            if (ApplyRedex(t) != null)
                res.Add(new List<int>(path));
            if (t is App a)
            {
                Walk(a.Func, new List<int>(path) { 0 });
                Walk(a.Arg, new List<int>(path) { 1 });
            }
        }

        Walk(term, new List<int>());
        return res;
    }

    // Reduit le redex situe au chemin indique (0 = func, 1 = arg).
    static Term ReplacePath(Term term, List<int> path)
    {
        if (path.Count == 0)
            return ApplyRedex(term) ?? throw new InvalidOperationException("pas de redex a la racine");
        var a = (App)term;
        List<int> rest = path.GetRange(1, path.Count - 1);
        if (path[0] == 0)
            return new App(ReplacePath(a.Func, rest), a.Arg);
        return new App(a.Func, ReplacePath(a.Arg, rest));
    }

    // --- Abstraction de bracket : lambda -> SKI ------------------------------

    // Traduit \var. body en combinateurs S/K/I (abstraction de bracket).
    // Schema : [x.x] = I ; [x.c] = K c (c sans x) ; [x.(M N)] = S [x.M] [x.N].
    // Traduction pure : les optimisations de Curry ne sont PAS appliquees, la
    // croissance du terme traduit est justement l'objet de la mesure.
    public static Term Lam(string v, Term body) => Abstract(v, body);

    static Term Abstract(string x, Term term)
    {
        if (!FreeVars(term).Contains(x))
            return new App(Term.K, term);
        if (term is Var bv && bv.Name == x)
            return Term.I;
        if (term is App ba)
            return new App(new App(Term.S, Abstract(x, ba.Func)), Abstract(x, ba.Arg));
        throw new ArgumentException("terme non abstrait de variable libre inattendue");
    }

    static HashSet<string> FreeVars(Term term)
    {
        if (term is Var v)
            return new HashSet<string> { v.Name };
        var a = (App)term;
        HashSet<string> res = FreeVars(a.Func);
        res.UnionWith(FreeVars(a.Arg));
        return res;
    }

    // --- Entiers de Church ---------------------------------------------------

    // Entier de Church n en S/K pur : \f x. f^n x, via bracket.
    public static Term Church(int n)
    {
        var f = new Var("f");
        var x = new Var("x");
        Term body = x;
        for (int i = 0; i < n; i++)
            body = new App(f, body);
        return Abstract("f", Abstract("x", body));
    }

    // Reconvertit un entier de Church en entier : application concrete du terme
    // abstrait, puis comptage de f (f (... x)).
    public static int? ChurchToInt(Term term, int maxSteps = 100000)
    {
        Term applied = new App(new App(term, new Var("f")), new Var("x"));
        (Term? nf, int _, List<Term>? _) = ReduceTerm(applied, "normal", maxSteps);
        if (nf == null)
            return null;
        return CountApplications(nf);
    }

    static int? CountApplications(Term term)
    {
        int count = 0;
        Term current = term;
        while (current is App a && a.Func is Var fv && fv.Name == "f")
        {
            count++;
            current = a.Arg;
        }

        return current is Var cv && cv.Name == "x" ? count : null;
    }

    // --- Systeme de tag de Post ---------------------------------------------

    // Un pas du systeme de tag : supprimer nu elements en tete, puis ajouter en
    // queue le bloc associe au premier element supprime. Null si le pas n'est pas
    // applicable (chaine plus courte que nu : le systeme est arrete).
    public static string? TagStep(string state)
    {
        if (state.Length < PostNu)
            return null;
        char dropped = state[0];
        if (!PostRules.TryGetValue(dropped, out string? block))
            return null;
        return state.Substring(PostNu) + block;
    }

    // Evolution complete : renvoie (trace, verdict).
    // Verdicts : 'halted' (chaine < nu), 'cycle' (etat deja vu),
    // 'unbounded' (borne de pas atteinte -- ni arret ni cycle constates).
    public static (List<string> Trace, string Verdict) TagEvolve(string graine, int maxSteps = 100000)
    {
        var trace = new List<string> { graine };
        var seen = new Dictionary<string, int> { [graine] = 0 };
        string current = graine;
        for (int step = 1; step <= maxSteps; step++)
        {
            string? nxt = TagStep(current);
            if (nxt == null)
                return (trace, "halted");
            trace.Add(nxt);
            if (seen.ContainsKey(nxt))
                return (trace, "cycle");
            seen[nxt] = step;
            current = nxt;
        }

        return (trace, "unbounded");
    }

    // Blocs de `width` elements qui n'apparaissent JAMAIS dans la trace observee.
    // Portee : la trace, pas le systeme entier.
    public static List<string> FirstMissingBlocks(IReadOnlyList<string> trace, int width = 4)
    {
        var seen = new HashSet<string>();
        foreach (string state in trace)
            for (int i = 0; i + width <= state.Length; i++)
                seen.Add(state.Substring(i, width));

        // L'ordre par defaut de Comparer<char> est ordinal et fournit le
        // comparateur IComparer<char> attendu par LINQ.
        char[] alphabet = trace
            .SelectMany(s => s.ToCharArray())
            .Distinct()
            .OrderBy(c => c)
            .ToArray();
        var missing = new List<string>();
        if (alphabet.Length == 0)
            return missing;

        var idx = new int[width];
        while (true)
        {
            string block = new string(idx.Select(i => alphabet[i]).ToArray());
            if (!seen.Contains(block))
                missing.Add(block);
            int p = width - 1;
            while (p >= 0)
            {
                idx[p]++;
                if (idx[p] < alphabet.Length)
                    break;
                idx[p] = 0;
                p--;
            }

            if (p < 0)
                break;
        }

        return missing;
    }
}

Combinateurs (S/K/I + tag de Post) : port manuel natif en C#


Regles : S f g x = f x (g x) | K x y = x | I x = x


## 1. Trois lettres : la syntaxe minimale du calcul

Le 7 decembre 1920, Moses Schonfinkel presente a Gottingen les combinateurs ; l'article de 11
pages qui suit (*Uber die Bausteine der mathematischen Logik*, 1924) introduit deux regles et
une seule operation -- l'application. Wolfram resume ce geste : parvenir a concevoir "that it's
even conceivable that everything can be defined in terms of them is a **supreme achievement of
a kind of abstract reductionism**" (*A Centennial View*, section *Combinators Arrive*).

Toute la syntaxe tient dans ce tableau :

| Combinateur | Regle de reduction | Lecture |
|---|---|---|
| `S f g x` | `-> f x (g x)` | distribue l'argument aux deux branches |
| `K x y` | `-> x` | selectionne et oublie |
| `I x` | `-> x` | identite -- **derivable** : `S K K x -> x` |

La cellule suivante reduit `S K K x` pas a pas avec l'organe : le combinateur I n'est pas un
primitif, c'est un theoreme de deux pas.

In [2]:
#nullable enable
// Illustration 1 — réduction de S K K x en ordre normal.
{
    Term terme = Combinateurs.ParseSki("S K K x");
    var resultat = Combinateurs.ReduceTerm(terme, "normal", 100, true);
    var trace = resultat.Trace ?? throw new InvalidOperationException("Trace absente");
    for (int i = 0; i < trace.Count; i++)
        Console.WriteLine($"pas {i}: {Combinateurs.ToStr(trace[i])}");
    Console.WriteLine($"forme normale : {(resultat.FormeNormale is null ? "aucune" : Combinateurs.ToStr(resultat.FormeNormale))} | pas : {resultat.Pas}");
    string[] attendu = { "S K K x", "K x (K x)", "x" };
    if (!trace.Select(Combinateurs.ToStr).SequenceEqual(attendu) || resultat.Pas != 2 || resultat.FormeNormale is null || Combinateurs.ToStr(resultat.FormeNormale) != "x")
        throw new InvalidOperationException("Écart de parité : réduction de S K K x");
}

pas 0: S K K x


pas 1: K x (K x)


pas 2: x


forme normale : x | pas : 2


**Lecture du resultat** : `S K K x -> K x (K x) -> x`. Deux lettres suffisent ; la troisieme
(I) s'obtient. C'est ce constat qui fonde le **S Combinator Challenge** (8 juin 2021, prix de
$20 000) : Wolfram y demande si **S seul** -- sans K -- est encore universel, decrivant Schonfinkel
comme ayant fourni "the first explicit example of a system capable of what we now call universal
computation" (section *Hiding in Plain Sight for a Century?*). La question du minimum absolu
reste ouverte un siecle apres : c'est le fil rouge de ce pli.

## 2. Ordre normal contre ordre applicatif : la strategie n'est pas cosmetique

Deux strategies classiques de reduction :

- **ordre normal** : reduire le redex le plus externe-gauche d'abord -- le theoreme de
  standardisation garantit qu'il trouve la forme normale quand elle existe ;
- **ordre applicatif** : evaluer les arguments d'abord -- la strategie des langages usuels, qui
  peut diverger sur un terme qui possede pourtant une forme normale.

Le temoin canonique est `K I OMEGA` (ou `OMEGA = S I I (S I I)` est le terme qui se reduit en
lui-meme, section 3) : la regle K n'a jamais besoin de son second argument.

In [3]:
#nullable enable
// Illustration 2 — les deux stratégies sur K I OMEGA.
{
    Term terme = new App(new App(Term.K, Term.I), Term.Omega);
    var normale = Combinateurs.ReduceTerm(terme, "normal", 500);
    var applicative = Combinateurs.ReduceTerm(terme, "applicative", 500);
    string texteNormal = normale.FormeNormale is null ? "diverge" : Combinateurs.ToStr(normale.FormeNormale);
    string texteApplicatif = applicative.FormeNormale is null ? "diverge" : Combinateurs.ToStr(applicative.FormeNormale);
    Console.WriteLine($"ordre normal    : {texteNormal} ({normale.Pas} pas)");
    Console.WriteLine($"ordre applicatif: {texteApplicatif} (borne {applicative.Pas} pas atteinte)");
    if (texteNormal != "I" || normale.Pas != 1 || applicative.FormeNormale is not null || applicative.Pas != 500)
        throw new InvalidOperationException("Écart de parité : stratégies de réduction");
}

ordre normal    : I (1 pas)


ordre applicatif: diverge (borne 500 pas atteinte)


**Lecture du resultat** : le **meme terme**, la **meme regle K** -- et deux destins. En ordre
normal, `K I OMEGA -> I` en un pas : l'argument jamais utilise n'est jamais evalue. En ordre
applicatif, la machine s'enfonce dans OMEGA et ne revient pas. La paresse n'est pas une
optimisation : c'est une difference **semantique** -- certains calculs n'existent que si l'on
s'interdit d'en faire certains autres.

## 3. OMEGA : la divergence pure

Le terme `S I I (S I I)` est le point fixe de la reduction : chaque pas produit un terme qui se
reduit encore. Wolfram note que, contrairement aux automates cellulaires ou aux machines de
Turing, les combinateurs ont "a certain immediate structural complexity, with their elaborate
tree structures" (*A Centennial View*, section *What Is Their Mathematics?*) -- la divergence
elle-meme y est structurale, pas un simple non-terminus.

In [4]:
#nullable enable
// Illustration 3 — OMEGA : la trace grossit, et aucune forme normale n'est atteinte.
Console.WriteLine($"terme initial : {Combinateurs.ToStr(Term.Omega)}");

(Term? formeNormale, int pas, List<Term>? trace) = Combinateurs.ReduceTerm(Term.Omega, "normal", 3, true);
if (trace != null)
{
    for (int i = 0; i < trace.Count; i++)
        Console.WriteLine($"pas {i}: {Combinateurs.ToStr(trace[i])}");
}

Console.WriteLine($"etats dans la trace : {trace?.Count.ToString() ?? "0"} (terme initial + pas demandes)");

// « diverge » reprend le libellé Python : le test constate seulement
// qu'aucune forme normale n'est atteinte en 200 pas. Une preuve de
// divergence doit établir séparément que la réduction se poursuit indéfiniment.
(Term? nf200, int pas200, List<Term>? _) = Combinateurs.ReduceTerm(Term.Omega, "normal", 200);
string verdict = nf200 is null ? "diverge -- aucune forme normale atteinte" : Combinateurs.ToStr(nf200);
Console.WriteLine($"apres 200 pas : {verdict}");

// Contrôle borné : aucune forme normale atteinte en 200 pas d'ordre normal.
bool conforme = nf200 is null && pas200 == 200;
Console.WriteLine($"apres 200 pas : forme normale atteinte = {(nf200 is null ? "non" : "oui")} | pas_effectues = {pas200} | pas attendus = 200 | {(conforme ? "CONFORME" : "ÉCART")}");
if (!conforme)
    throw new Exception("Observation d'OMEGA à 200 pas non conforme à la mesure attendue");

terme initial : S I I (S I I)


pas 0: S I I (S I I)


pas 1: I (S I I) (I (S I I))


pas 2: S I I (I (S I I))


pas 3: I (I (S I I)) (I (I (S I I)))


etats dans la trace : 4 (terme initial + pas demandes)


apres 200 pas : diverge -- aucune forme normale atteinte


apres 200 pas : forme normale atteinte = non | pas_effectues = 200 | pas attendus = 200 | CONFORME


**Lecture du resultat** : la trace grossit a chaque pas -- le terme se replie sur sa propre
structure sans jamais s'aplatir. C'est le materiau brut de l'indecidabilite : Church et Kleene
introduisent la lambda-definissabilite vers 1934, Turing les machines en 1936, et "that -- in
effect by thinking about having it compile itself -- Hilbert's Entscheidungsproblem couldn't be
solved" (*A Centennial View*, section *Godel's Theorem and Computability*). Le meme essay
rapporte que Church demontre l'indecidabilite de l'equivalence des expressions lambda avant que
Turing n'etablisse la correspondance avec ses machines (section *Lambda Calculus*). Un moteur
assez riche pour tout calculer est assez riche pour ne rien garantir.

## 4. L'abstraction de bracket : eliminer les variables

Le pont entre le lambda-calcul de Church et les combinateurs est l'**abstraction de bracket** :
traduire `\x. M` en S/K/I par trois schemas -- `[x.x] = I`, `[x.c] = K c` (c sans x),
`[x.(M N)] = S [x.M] [x.N]`. Wolfram releve que Church, "on the very first page he defines
lambda, he's referencing Schonfinkel's combinator paper" (*A Centennial View*, section *Lambda
Calculus*) : les deux formalismes sont nes enchevetres.

L'organe `Combinateurs.Lam` implemente la traduction **pure**, sans les optimisations de Curry
(combinateurs B, C...) : la croissance du terme traduit est justement l'objet de la mesure --
optimiser la traduction, ce serait maquiller le prix de l'elimination des variables.

In [5]:
#nullable enable
// Illustration 4 — abstraction de bracket et taille des entiers de Church.
{
    int Taille(Term t) => t is App a ? 1 + Taille(a.Func) + Taille(a.Arg) : 1;
    Term[] traductions = {
        Combinateurs.Lam("x", new Var("x")),
        Combinateurs.Lam("x", new Var("c")),
        Combinateurs.Lam("x", Combinateurs.ParseSki("f x (g x)"))
    };
    string[] etiquettes = { @"\x.x          ", @"\x.c          ", @"\x.f x (g x)  " };
    string[] attendus = { "I", "K c", "S (S (K f) I) (S (K g) I)" };
    for (int i = 0; i < traductions.Length; i++)
    {
        string mesure = Combinateurs.ToStr(traductions[i]);
        Console.WriteLine($"{etiquettes[i]} -> {mesure}");
        if (mesure != attendus[i]) throw new InvalidOperationException("Écart de parité : abstraction");
    }
    int[] taillesAttendues = { 19, 35, 51, 67, 83 };
    for (int n = 1; n <= 5; n++)
    {
        int taille = Taille(Combinateurs.Church(n));
        Console.WriteLine($"church({n}) en S/K pur : {taille,4} noeuds");
        if (taille != taillesAttendues[n - 1]) throw new InvalidOperationException($"Écart de parité : taille de church({n})");
    }
}

\x.x           -> I


\x.c           -> K c


\x.f x (g x)   -> S (S (K f) I) (S (K g) I)


church(1) en S/K pur :   19 noeuds


church(2) en S/K pur :   35 noeuds


church(3) en S/K pur :   51 noeuds


church(4) en S/K pur :   67 noeuds


church(5) en S/K pur :   83 noeuds


**Lecture du résultat** : `\x. f x (g x)` devient `S (S (K f) I) (S (K g) I)` : chaque variable éliminée laisse une trace dans la structure du terme. Dans la traduction pure de ce moteur, la taille de `church(n)` croît linéairement. Pour mesurer cette croissance, comptez aussi bien les feuilles que les applications, puis comparez deux tailles consécutives. Les optimisations de Curry (B, C...) visent précisément à réduire ce coût.

## 5. Les entiers de Church : l'arithmetique sans nombres

Un entier n est la fonction qui applique n fois son argument : `church(n) = \f x. f^n x`.
L'addition, la multiplication et l'exponentiation sont alors des **compositions** : ajouter,
c'est enchainer ; multiplier, c'est iterer l'iteration ; elever a une puissance, c'est iterer
la multiplication. Le carnet construit ces combinateurs par abstraction de bracket puis les
execute -- l'arithmetique est reelle, pas recitee.

In [6]:
#nullable enable
// Illustration 5 — arithmétique sur les entiers de Church.
{
    Term deux = Combinateurs.Church(2);
    Term trois = Combinateurs.Church(3);
    string ecriture = Combinateurs.ToStr(deux);
    Console.WriteLine($"church(2) = {ecriture}");
    int?[] allerRetour = Enumerable.Range(0, 8).Select(n => Combinateurs.ChurchToInt(Combinateurs.Church(n))).ToArray();
    Console.WriteLine($"round-trip 0..7 : [{string.Join(", ", allerRetour)}]");
    Term plus = Combinateurs.Lam("m", Combinateurs.Lam("n", Combinateurs.Lam("f", Combinateurs.Lam("x", Combinateurs.ParseSki("m f (n f x)")))));
    Term mult = Combinateurs.Lam("m", Combinateurs.Lam("n", Combinateurs.Lam("f", Combinateurs.Lam("x", Combinateurs.ParseSki("n (m f) x")))));
    Term exp = Combinateurs.Lam("m", Combinateurs.Lam("n", Combinateurs.ParseSki("n m")));
    int? somme = Combinateurs.ChurchToInt(new App(new App(plus, deux), trois));
    int? produit = Combinateurs.ChurchToInt(new App(new App(mult, deux), trois));
    int? puissance = Combinateurs.ChurchToInt(new App(new App(exp, deux), trois));
    Console.WriteLine($"2 + 3 = {somme} | 2 * 3 = {produit} | 2 ^ 3 = {puissance}");
    if (ecriture != "S (S (K S) (S (K K) I)) (S (S (K S) (S (K K) I)) (K I))" ||
        !allerRetour.SequenceEqual(Enumerable.Range(0, 8).Select(n => (int?)n)) || somme != 5 || produit != 6 || puissance != 8)
        throw new InvalidOperationException("Écart de parité : arithmétique de Church");
}

church(2) = S (S (K S) (S (K K) I)) (S (S (K S) (S (K K) I)) (K I))


round-trip 0..7 : [0, 1, 2, 3, 4, 5, 6, 7]


2 + 3 = 5 | 2 * 3 = 6 | 2 ^ 3 = 8


**Lecture du resultat** : `5`, `6`, `8` -- calcules par `S` et `K` seuls, sans aucun entier
natif jusqu'a la conversion finale. C'est le sens operationnel de la these d'universalite :
non pas un slogan, mais le fait que trois lignes de composition remplacent les tables
d'operation. La lecture `n (m f)` pour la multiplication -- iterer `m` fois, puis iterer ce
resultat `n` fois -- est le moment exact ou l'exposant devient une iteration d'iterations.

## 6. Church-Rosser empirique : mille chemins, une seule forme normale

Le **theoreme de Church-Rosser** (confluence) etablit que si un terme a deux formes normales,
elles sont egales : l'ordre de reduction ne change pas le resultat, seulement le chemin. Ce
carnet n'en donne **pas une preuve** -- il en donne le **temoin empirique** : depuis un meme
terme, reduire un redex **aleatoire** a chaque pas, 30 fois, et compter les formes normales
distinctes.

In [7]:
#nullable enable
// Illustration 6 — Church-Rosser empirique : 30 ordres aléatoires.
var rng = new Random(42);
Term terme = new App(new App(Combinateurs.Church(5), new Var("f")), new Var("x"));
var formes = new HashSet<string>(StringComparer.Ordinal);
var pasVus = new List<int>();
int sansFormeNormale = 0;

for (int i = 0; i < 30; i++)
{
    (Term? nf, int nPas) = Combinateurs.RandomReduce(terme, rng, 20000);
    pasVus.Add(nPas);
    if (nf is null)
        sansFormeNormale++;
    else
        formes.Add(Combinateurs.ToStr(nf));
}

int minPas = pasVus.Min();
int maxPas = pasVus.Max();
int mediane = pasVus.OrderBy(x => x).ElementAt(15);
string[] formesTriees = formes.OrderBy(x => x, StringComparer.Ordinal).ToArray();
Console.WriteLine($"formes normales distinctes : {formesTriees.Length}");
foreach (string forme in formesTriees)
    Console.WriteLine($"    {forme}");
Console.WriteLine($"pas de reduction : min {minPas} | max {maxPas} | median {mediane}");

// Banc de parité indépendant des stubs. Les références Python sont des littéraux.
var ecartsParite = new List<string>();
void ComparerParite(string libelle, string mesure, string reference)
{
    bool conforme = string.Equals(mesure, reference, StringComparison.Ordinal);
    Console.WriteLine($"{libelle} : C# = {mesure} | Python = {reference} | {(conforme ? "CONFORME" : "ÉCART")}");
    if (!conforme)
        ecartsParite.Add(libelle);
}

// Un certificat vérifie une affirmation figée, sans produire la synthèse de l'exercice.
void VerifierCertificat(string libelle, string reference, bool verifie)
{
    Console.WriteLine($"{libelle} : Python = {reference} | certificat C# = {verifie} | {(verifie ? "CONFORME" : "ÉCART")}");
    if (!verifie)
        ecartsParite.Add(libelle);
}

string NotationListe(IEnumerable<string> elements) => "[" + string.Join(", ", elements.Select(e => "'" + e + "'")) + "]";
string TexteEntier(int? n) => n?.ToString(CultureInfo.InvariantCulture) ?? "aucun";

string bilanConfluence = $"{formesTriees.Length} forme(s) [{string.Join(", ", formesTriees)}] ; min/max/médiane {minPas}/{maxPas}/{mediane} ; {sansFormeNormale} sans forme normale";
ComparerParite("§6 — Church-Rosser", bilanConfluence, "1 forme(s) [f (f (f (f (f x))))] ; min/max/médiane 42/42/42 ; 0 sans forme normale");

// §1 : certificat de trace Python. Chaque pas est testé sur une entrée figée ;
// aucun résultat d'un test ne sert d'entrée au suivant.
string[] temoinSkk = { "S K K x", "K x (K x)", "x" };
bool transitionsSkkValides = temoinSkk.Zip(temoinSkk.Skip(1), (depart, attendu) =>
    Combinateurs.ReduceOnce(Combinateurs.ParseSki(depart), "normal") is Term successeur
    && string.Equals(Combinateurs.ToStr(successeur), attendu, StringComparison.Ordinal)).All(ok => ok);
VerifierCertificat("§1 — trace S K K x", NotationListe(temoinSkk), transitionsSkkValides);
VerifierCertificat("§1 — forme normale et pas", "x | pas : 2",
    transitionsSkkValides && temoinSkk[0] == "S K K x" && temoinSkk[^1] == "x"
    && temoinSkk.Length - 1 == 2 && temoinSkk.Length - 1 < 100
    && Combinateurs.IsNormalForm(Combinateurs.ParseSki(temoinSkk[^1])));

// §2 : vérifier séparément deux propriétés locales, sans exécuter l'exercice.
// En ordre normal, K efface son second argument à la racine, quel que soit
// le terme substitué au témoin libre. I est déjà une forme normale.
Term contexteTemoin = Combinateurs.ParseSki("K I z");
bool effacement = Combinateurs.ApplyRedex(contexteTemoin) is Term retenu
    && object.ReferenceEquals(retenu, Term.I) && Combinateurs.IsNormalForm(retenu);
VerifierCertificat("§2 — ordre normal", "I (1 pas)", effacement);

// En ordre applicatif, K I est irréductible et le redex de l'argument
// est prioritaire sur celui de la racine. Un cycle de l'argument seul
// suffit donc pour toute borne finie, dont les 500 pas du protocole.
var cycleOmega = new[] { "S I I (S I I)", "I (S I I) (I (S I I))", "S I I (I (S I I))" };
bool cycleOmegaVerifie = Combinateurs.IsNormalForm(Combinateurs.ParseSki("K I"))
    && Combinateurs.ToStr(Term.Omega) == cycleOmega[0]
    && Enumerable.Range(0, cycleOmega.Length).All(i =>
        Combinateurs.ReduceOnce(Combinateurs.ParseSki(cycleOmega[i]), "applicative") is Term successeur
        && Combinateurs.ToStr(successeur) == cycleOmega[(i + 1) % cycleOmega.Length]);
VerifierCertificat("§2 — ordre applicatif", "diverge (borne 500 pas atteinte)", cycleOmegaVerifie);

// §3 : comparer les mesures de la validation OMEGA déjà exécutée.
ComparerParite("§3 — OMEGA, terme initial", Combinateurs.ToStr(Term.Omega), "S I I (S I I)");
ComparerParite("§3 — OMEGA, trace à trois pas", trace is null ? "aucune" : NotationListe(trace.Select(Combinateurs.ToStr)),
    "['S I I (S I I)', 'I (S I I) (I (S I I))', 'S I I (I (S I I))', 'I (I (S I I)) (I (I (S I I)))']");
ComparerParite("§3 — OMEGA, verdict à 200 pas", $"{pas200} pas | {verdict}", "200 pas | diverge -- aucune forme normale atteinte");

// §4 : tester le comportement des traductions Python prises comme entrées
// figées. Ce contrôle n'appelle pas Lam et ne reconstitue aucun corps à abstraire.
var temoinsBracket = new (string Etiquette, string Reference, string ImageSurZ)[]
{
    (@"\x.x", "I", "z"),
    (@"\x.c", "K c", "c"),
    (@"\x.f x (g x)", "S (S (K f) I) (S (K g) I)", "f z (g z)")
};
foreach (var temoin in temoinsBracket)
{
    string application = $"({temoin.Reference}) z";
    var observation = Combinateurs.ReduceTerm(Combinateurs.ParseSki(application), "normal", 100);
    VerifierCertificat($"§4 — {temoin.Etiquette}, comportement de la référence", temoin.Reference,
        observation.FormeNormale is Term image
        && string.Equals(Combinateurs.ToStr(image), temoin.ImageSurZ, StringComparison.Ordinal));
}

// §5 : vérifier séparément l'encodage et le décodage contre les mêmes témoins.
// Chaque écriture de référence est stockée sous forme compacte : un fragment
// répété autour du littéral K I. Le décodeur ne reçoit jamais la sortie de l'encodeur.
ComparerParite("§5 — church(2)", Combinateurs.ToStr(Combinateurs.Church(2)),
    "S (S (K S) (S (K K) I)) (S (S (K S) (S (K K) I)) (K I))");
const string fragmentChurchPython = "S (S (K S) (S (K K) I)) (";
string EcritureChurchTemoin(int repetitions) =>
    string.Concat(Enumerable.Repeat(fragmentChurchPython, repetitions)) + "K I" + new string(')', repetitions);
int[] entiersChurchPython = { 0, 1, 2, 3, 4, 5, 6, 7 };
bool[] ecrituresIdentiques = entiersChurchPython.Select(n =>
    string.Equals(Combinateurs.ToStr(Combinateurs.Church(n)), EcritureChurchTemoin(n), StringComparison.Ordinal)).ToArray();
bool encodageChurchValide = ecrituresIdentiques.All(identique => identique);
bool decodageChurchValide = entiersChurchPython.All(n =>
    Combinateurs.ChurchToInt(Combinateurs.ParseSki(EcritureChurchTemoin(n))) == n);
VerifierCertificat("§5 — round-trip 0..7", "[0, 1, 2, 3, 4, 5, 6, 7]", encodageChurchValide && decodageChurchValide);

// §4 : les tailles restent des références Python. L'égalité des écritures
// intégralement parenthésées selon la convention du moteur fixe le même
// arbre d'application, sans mesurer sa taille ni compter ses nœuds.
int[] taillesPython = { 19, 35, 51, 67, 83 };
for (int n = 1; n <= taillesPython.Length; n++)
    VerifierCertificat($"§4 — church({n}) en S/K pur, identité de l'arbre",
        $"{taillesPython[n - 1],4} noeuds", ecrituresIdentiques[n]);

// L'arithmétique teste trois termes fermés par composition, sans définir
// plus, mult ou exp par bracket.
Term AppliquerTemoin(Term tete, params Term[] args) => args.Aggregate(tete, (func, arg) => new App(func, arg));
Term composition = Combinateurs.B;
Term temoinDeux = Combinateurs.Church(2);
Term temoinTrois = Combinateurs.Church(3);
Term temoinSomme = AppliquerTemoin(Term.S, AppliquerTemoin(composition, composition, temoinDeux), temoinTrois);
Term temoinProduit = AppliquerTemoin(composition, temoinDeux, temoinTrois);
Term temoinPuissance = AppliquerTemoin(temoinTrois, AppliquerTemoin(composition, temoinDeux), Term.I);
ComparerParite("§5 — 2 + 3", TexteEntier(Combinateurs.ChurchToInt(temoinSomme)), "5");
ComparerParite("§5 — 2 * 3", TexteEntier(Combinateurs.ChurchToInt(temoinProduit)), "6");
ComparerParite("§5 — 2 ^ 3", TexteEntier(Combinateurs.ChurchToInt(temoinPuissance)), "8");

// §7 : vérifier des certificats figés avec le seul pas élémentaire.
// Cette observation bornée ne cherche pas le premier cycle et ne renvoie
// ni verdict ni statistiques. Une répétition ne l'arrête pas.
IEnumerable<string> ObserverPas(string depart, int borne)
{
    string? etatObserve = depart;
    for (int p = 0; p <= borne && etatObserve is string etat; p++)
    {
        yield return etat;
        if (p < borne)
            etatObserve = Combinateurs.TagStep(etat);
    }
}

bool OccupePositions(IReadOnlyList<string> etats, int indice, params string[] temoin) =>
    indice >= 0 && indice + temoin.Length <= etats.Count
    && Enumerable.Range(0, temoin.Length).All(i => etats[indice + i] == temoin[i]);

bool EncadrementExact(IEnumerable<string> etats, int bas, int haut) =>
    etats.All(e => e.Length >= bas && e.Length <= haut)
    && etats.Any(e => e.Length == bas) && etats.Any(e => e.Length == haut);

bool PremierRetourAuRang(IReadOnlyList<string> etats, int rang, int borne) =>
    rang <= borne && etats.Count == rang + 1
    && etats.Take(rang).Distinct(StringComparer.Ordinal).Count() == rang
    && etats.Take(rang).Contains(etats[rang], StringComparer.Ordinal);

const int borneCourtePost = 100000;
var preuve10 = ObserverPas("10", 1).ToArray();
VerifierCertificat("§7 — graine '10'", "halted | etats: 1 | longueurs min/max: 2/2",
    preuve10.Length == 1 && EncadrementExact(preuve10, 2, 2));
VerifierCertificat("§7 — '10', tête", "['10']", OccupePositions(preuve10, 0, "10"));
VerifierCertificat("§7 — '10', fin", "['10']", OccupePositions(preuve10, 0, "10"));

var traceBlocsPost = ObserverPas("1001010010000", 32).ToArray();
VerifierCertificat("§7 — graine '1001010010000'", "cycle | etats: 33 | longueurs min/max: 8/16",
    PremierRetourAuRang(traceBlocsPost, 32, borneCourtePost) && EncadrementExact(traceBlocsPost, 8, 16));
VerifierCertificat("§7 — '1001010010000', tête", "['1001010010000', '10100100001101', '001000011011101', '00001101110100']",
    OccupePositions(traceBlocsPost, 0, "1001010010000", "10100100001101", "001000011011101", "00001101110100"));
VerifierCertificat("§7 — '1001010010000', fin", "['0000110111011101', '011011101110100']",
    OccupePositions(traceBlocsPost, 31, "0000110111011101", "011011101110100"));

var preuve110101 = ObserverPas("110101", 21).ToArray();
VerifierCertificat("§7 — graine '110101'", "cycle | etats: 22 | longueurs min/max: 6/16",
    PremierRetourAuRang(preuve110101, 21, borneCourtePost) && EncadrementExact(preuve110101, 6, 16));
VerifierCertificat("§7 — '110101', tête", "['110101', '1011101', '11011101', '111011101']",
    OccupePositions(preuve110101, 0, "110101", "1011101", "11011101", "111011101"));
VerifierCertificat("§7 — '110101', fin", "['0000110111011101', '011011101110100']",
    OccupePositions(preuve110101, 20, "0000110111011101", "011011101110100"));

const string seedPost31 = "1001001001001001001001001001001";
var preuvePost31 = ObserverPas(seedPost31, 5000).ToArray();
// Une répétition dans un système déterministe garantit que les états
// ultérieurs ne peuvent pas introduire de nouvelle longueur.
VerifierCertificat("§7 — graine périodique (31 éléments)", "cycle | longueur max atteinte : 208",
    seedPost31.Length == 31 && preuvePost31.GroupBy(s => s, StringComparer.Ordinal).Any(g => g.Skip(1).Any())
    && preuvePost31.All(s => s.Length <= 208) && preuvePost31.Any(s => s.Length == 208));

// §8 : deux oracles sans fonction de substitution ni simulation de file.
// Thue-Morse est donné par la parité du nombre de bits à 1 de l'indice.
// Pour Fibonacci, W0 = 0, W1 = 01 et Wn = W(n-1) W(n-2).
var blocsPost = Combinateurs.FirstMissingBlocks(traceBlocsPost, 4);
ComparerParite("§8 — blocs de largeur 4 absents de la trace", NotationListe(blocsPost), "['1100', '1111']");
ComparerParite("§8 — 1111 absent", blocsPost.Contains("1111") ? "True" : "False", "True");

const int genTmAffichee = 6;
const int genFiboAffichee = 7;
int genTmRecherche = genTmAffichee + 4;
int genFiboRecherche = genFiboAffichee + 6;
string MotTmOracle(int n) => new string(Enumerable.Range(0, 1 << n)
    .Select(i => (System.Numerics.BitOperations.PopCount((uint)i) & 1) == 0 ? '0' : '1').ToArray());
var motsFiboOracle = new List<string> { "0", "01" };
while (motsFiboOracle.Count <= genFiboRecherche)
    motsFiboOracle.Add(motsFiboOracle[^1] + motsFiboOracle[^2]);
string motTmAffiche = MotTmOracle(genTmAffichee);
string motFiboAffiche = motsFiboOracle[genFiboAffichee];
ComparerParite("§8 — Thue-Morse", $"({motTmAffiche.Length}) : {motTmAffiche}",
    "(64) : 0110100110010110100101100110100110010110011010010110100110010110");
ComparerParite("§8 — Fibonacci", $"({motFiboAffiche.Length}) : {motFiboAffiche}", "(34) : 0100101001001010010100100101001001");

// À r = 1, les lettres sont lues dans la concaténation des générations.
// Après p lectures, la fenêtre active commence à p ; sa longueur vaut
// un plus le nombre de lettres déjà lues qui donnent un bloc de longueur deux.
const int pasOracleR1 = 20;
int[] PositionsFenetres(string generations, string recherche, Func<char, bool> produitDeux)
{
    return Enumerable.Range(0, pasOracleR1 + 1).Select(p =>
    {
        int longueur = 1 + generations.Take(p).Count(produitDeux);
        return recherche.IndexOf(generations.Substring(p, longueur), StringComparison.Ordinal);
    }).ToArray();
}
string generationsTm = string.Concat(Enumerable.Range(0, genTmRecherche + 1).Select(MotTmOracle));
string generationsFibo = string.Concat(motsFiboOracle);
int[] positionsTm = PositionsFenetres(generationsTm, MotTmOracle(genTmRecherche), _ => true);
int[] positionsFibo = PositionsFenetres(generationsFibo, motsFiboOracle[genFiboRecherche], c => c == '0');
ComparerParite("§8 — nombre d'états r=1", TexteEntier(positionsTm.Length), "21");
ComparerParite("§8 — préfixes de Thue-Morse", TexteEntier(positionsTm.Count(i => i == 0)), "5");
ComparerParite("§8 — facteurs de Thue-Morse", TexteEntier(positionsTm.Count(i => i >= 0)), "21");
ComparerParite("§8 — préfixes de Fibonacci", TexteEntier(positionsFibo.Count(i => i == 0)), "6");
ComparerParite("§8 — facteurs de Fibonacci", TexteEntier(positionsFibo.Count(i => i >= 0)), "21");

Console.WriteLine($"Attestation — écarts mesurés : {ecartsParite.Count}");
if (ecartsParite.Count != 0)
    throw new InvalidOperationException($"Écarts de parité : {string.Join(", ", ecartsParite)}");

formes normales distinctes : 1


    f (f (f (f (f x))))


pas de reduction : min 42 | max 42 | median 42


§6 — Church-Rosser : C# = 1 forme(s) [f (f (f (f (f x))))] ; min/max/médiane 42/42/42 ; 0 sans forme normale | Python = 1 forme(s) [f (f (f (f (f x))))] ; min/max/médiane 42/42/42 ; 0 sans forme normale | CONFORME


§1 — trace S K K x : Python = ['S K K x', 'K x (K x)', 'x'] | certificat C# = True | CONFORME


§1 — forme normale et pas : Python = x | pas : 2 | certificat C# = True | CONFORME


... (35 outputs left out)

§8 — préfixes de Fibonacci : C# = 6 | Python = 6 | CONFORME


§8 — facteurs de Fibonacci : C# = 21 | Python = 21 | CONFORME


Attestation — écarts mesurés : 0



(85,90): warning CS8604: Possible null reference argument for parameter 'source' in 'IEnumerable<string> Enumerable.Select<Term, string>(IEnumerable<Term> source, Func<Term, string> selector)'.



**Lecture du resultat -- et sa limite, dite**. Les 30 ordres convergent vers la meme forme
normale `f (f (f (f (f x))))` : c'est le temoin. Sur ce terme precis, le nombre de pas est meme
constant (42) : aucun redex du terme n'en duplique ni n'en detruit un autre -- une propriete de
ce terme, pas de la reduction en general (le redex d'OMEGA, lui, se duplique a chaque pas).

**Un temoin n'est pas une preuve** : trente ordres ne couvrent pas l'infinite des chemins. La
preuve formelle existe desormais : le lake
[`combinators_lean`](../../SymbolicAI/Lean/combinators_lean/) (pli 4, grain B) demontre
`confluence` -- toute paire de reductions `step*` issues d'un meme terme se rejoint -- et son
corollaire `unique_normal_form` : deux reductions exhaustives d'un meme terme aboutissent au
meme terme irreductible. La preuve invoque l'organe Mathlib `Relation.church_rosser` (forme de
Hindley-Rosen : le losange local de la reduction parallele entraine la confluence de sa cloture
reflexive-transitive), menee de style Tait-Martin-Lof sans lieurs -- ce que S/K/I rend bien plus
simple que le lambda-calcul. Ce carnet fournit l'instrument de mesure ; le lake fournit la
preuve verifiee par le noyau Lean.

## 7. Le probleme de tag de Post : la borne d'irreductibilite

En 1920-1921, Emil Post simplifie la logique de Principia Mathematica en transformations de
chaines, jusqu'a un systeme minimal : alphabet `{0, 1}`, a chaque pas **supprimer 3 elements
en tete** puis **ajouter en queue** le bloc associe au premier element supprime (`0 -> 00`,
`1 -> 1101`). La question : **toute graine mene-t-elle a l'arret** (chaine plus courte que 3),
a un cycle, ou a une croissance sans borne ?

Wolfram raconte que Post "began on it in the early summer of 1921, no doubt expecting that such
a simple-to-state problem would have a correspondingly simple solution" (section *[Despite]
Considerable Effort... [It Proved] Intractable*) -- et que lui-meme doit reporter, un siecle
plus tard : "even a century later I still haven't been able to answer this question" (section
*So Does It Always Halt?*).

In [8]:
#nullable enable
// Illustration 7 — évolutions du système de tag de Post.
{
    string Liste(IEnumerable<string> xs) => "[" + string.Join(", ", xs.Select(x => "'" + x + "'")) + "]";
    string[] graines = { "10", "1001010010000", "110101" };
    string[] verdicts = { "halted", "cycle", "cycle" };
    int[] comptes = { 1, 33, 22 };
    int[] minima = { 2, 8, 6 };
    int[] maxima = { 2, 16, 16 };
    string[] tetes = { "['10']", "['1001010010000', '10100100001101', '001000011011101', '00001101110100']", "['110101', '1011101', '11011101', '111011101']" };
    string[] fins = { "['10']", "['0000110111011101', '011011101110100']", "['0000110111011101', '011011101110100']" };
    for (int i = 0; i < graines.Length; i++)
    {
        var evolution = Combinateurs.TagEvolve(graines[i], 100000);
        int minimum = evolution.Trace.Min(s => s.Length);
        int maximum = evolution.Trace.Max(s => s.Length);
        string tete = Liste(evolution.Trace.Take(4));
        string fin = Liste(evolution.Trace.Skip(Math.Max(0, evolution.Trace.Count - 2)));
        string graineAffichee = "'" + graines[i] + "'";
        Console.WriteLine($"graine {graineAffichee,-18} -> {evolution.Verdict,-9} | etats: {evolution.Trace.Count,5} | longueurs min/max: {minimum}/{maximum}");
        Console.WriteLine($"   tete : {tete}");
        Console.WriteLine($"   fin  : {fin}");
        if (evolution.Verdict != verdicts[i] || evolution.Trace.Count != comptes[i] || minimum != minima[i] || maximum != maxima[i] || tete != tetes[i] || fin != fins[i])
            throw new InvalidOperationException($"Écart de parité : graine {graines[i]}");
    }
    var longue = Combinateurs.TagEvolve("1001001001001001001001001001001", 5000);
    int maxLong = longue.Trace.Max(s => s.Length);
    Console.WriteLine($"graine periodique (31 elements) -> {longue.Verdict} | longueur max atteinte : {maxLong}");
    if (longue.Verdict != "cycle" || maxLong != 208) throw new InvalidOperationException("Écart de parité : graine périodique");
}

graine '10'               -> halted    | etats:     1 | longueurs min/max: 2/2


   tete : ['10']


   fin  : ['10']


graine '1001010010000'    -> cycle     | etats:    33 | longueurs min/max: 8/16


   tete : ['1001010010000', '10100100001101', '001000011011101', '00001101110100']


   fin  : ['0000110111011101', '011011101110100']


graine '110101'           -> cycle     | etats:    22 | longueurs min/max: 6/16


   tete : ['110101', '1011101', '11011101', '111011101']


   fin  : ['0000110111011101', '011011101110100']


graine periodique (31 elements) -> cycle | longueur max atteinte : 208


**Lecture du resultat** : trois destins mesures -- **arret** immediat (`10` est plus courte que
le seuil), **cycle** (la graine de 13 elements de l'essai de Wolfram, et une graine de 6,
rejoignent le meme attracteur), **cycle apres croissance** (une graine periodique de 31 monte
jusqu'a 208 elements avant de retomber). Aucune graine testee ici ne croit sans borne -- mais
l'echantillon ne prouve rien : c'est exactement le statut du probleme.

Post lui-meme voyait plus loin que son echec : "I would have discovered Godel's theorem in
1921 -- if I had been Godel" (cite par Wolfram, meme section introductive). Et en 1961, Marvin
Minsky demontre qu'un systeme de tag **convenablement construit** peut calculer tout ce qu'une
machine de Turing calcule (section *[Despite] Considerable Effort...*) : le probleme de Post
n'est pas dur parce que le systeme est pauvre -- il est dur parce qu'il est deja riche.

## 8. Blocs inatteignables et r = 1 : ou passe la frontiere causale

Deux faits de structure, tous deux mesures par Wolfram (2021) :

1. **Le bloc `1111` est inatteignable** : "no sequence of the 00s and 1101s inserted by the rule
   can ever produce it" (section *The Basic Setup*). Une preuve de deux lignes sur les regles --
   mais une contrainte permanente sur tout l'espace d'etats.
2. **La suppression d'un seul element par pas (r = 1) tue la causalite** : "rules which delete
   only one element at each step cannot involve real communication (or causal connections)
   between different parts of the string, and must be equivalent to neighbor-independent
   substitution systems" -- les regles `0 -> 01, 1 -> 10` engendrent le mot de Thue-Morse,
   `0 -> 01, 1 -> 0` le mot de Fibonacci (section *Other Tag Systems*).

In [9]:
#nullable enable
// Illustration 8 — blocs absents, substitutions et système de tag à r = 1.
{
    string Liste(IEnumerable<string> xs) => "[" + string.Join(", ", xs.Select(x => "'" + x + "'")) + "]";
    var evolution = Combinateurs.TagEvolve("1001010010000");
    var manquants = Combinateurs.FirstMissingBlocks(evolution.Trace, 4);
    bool absent = manquants.Contains("1111");
    Console.WriteLine($"blocs de largeur 4 absents de la trace : {Liste(manquants)}");
    Console.WriteLine($"1111 absent : {absent}");
    var reglesTm = new Dictionary<char, string> { ['0'] = "01", ['1'] = "10" };
    var reglesFibo = new Dictionary<char, string> { ['0'] = "01", ['1'] = "0" };
    string Substitue(string s, IReadOnlyDictionary<char, string> regles) => string.Concat(s.Select(c => regles[c]));
    string Genere(string s, IReadOnlyDictionary<char, string> regles, int n)
    {
        for (int i = 0; i < n; i++) s = Substitue(s, regles);
        return s;
    }
    string thue = Genere("0", reglesTm, 6);
    string fibo = Genere("0", reglesFibo, 7);
    Console.WriteLine($"Thue-Morse ({thue.Length}) : {thue}");
    Console.WriteLine($"Fibonacci ({fibo.Length})  : {fibo}");
    string thueLong = Genere(thue, reglesTm, 4);
    string fiboLong = Genere(fibo, reglesFibo, 6);
    List<string> EvolueR1(string graine, IReadOnlyDictionary<char, string> regles, int pas)
    {
        var file = new Queue<char>(graine);
        var etats = new List<string> { new string(file.ToArray()) };
        for (int i = 0; i < pas; i++)
        {
            char tete = file.Dequeue();
            foreach (char c in regles[tete]) file.Enqueue(c);
            etats.Add(new string(file.ToArray()));
        }
        return etats;
    }
    var etatsTm = EvolueR1("0", reglesTm, 20);
    var etatsFibo = EvolueR1("0", reglesFibo, 20);
    int prefixesTm = etatsTm.Count(e => thueLong.StartsWith(e, StringComparison.Ordinal));
    int facteursTm = etatsTm.Count(e => thueLong.Contains(e, StringComparison.Ordinal));
    int prefixesFibo = etatsFibo.Count(e => fiboLong.StartsWith(e, StringComparison.Ordinal));
    int facteursFibo = etatsFibo.Count(e => fiboLong.Contains(e, StringComparison.Ordinal));
    Console.WriteLine($"sur {etatsTm.Count} etats r=1 :");
    Console.WriteLine($"  prefixes de Thue-Morse : {prefixesTm}");
    Console.WriteLine($"  facteurs de Thue-Morse : {facteursTm}");
    Console.WriteLine($"  prefixes de Fibonacci  : {prefixesFibo}");
    Console.WriteLine($"  facteurs de Fibonacci  : {facteursFibo}");
    if (!manquants.SequenceEqual(new[] { "1100", "1111" }) || !absent ||
        thue != "0110100110010110100101100110100110010110011010010110100110010110" ||
        fibo != "0100101001001010010100100101001001" || etatsTm.Count != 21 || etatsFibo.Count != 21 ||
        prefixesTm != 5 || facteursTm != 21 || prefixesFibo != 6 || facteursFibo != 21)
        throw new InvalidOperationException("Écart de parité : blocs et substitutions");
}

blocs de largeur 4 absents de la trace : ['1100', '1111']


1111 absent : True


Thue-Morse (64) : 0110100110010110100101100110100110010110011010010110100110010110


Fibonacci (34)  : 0100101001001010010100100101001001


sur 21 etats r=1 :


  prefixes de Thue-Morse : 5


  facteurs de Thue-Morse : 21


  prefixes de Fibonacci  : 6


  facteurs de Fibonacci  : 21


<null>

**Lecture du resultat** : a `r = 1`, les etats visites sont des **facteurs** (sous-mots) des
mots de Thue-Morse et de Fibonacci -- des suites parfaitement regulieres, generees par
substitution, sans aucune communication entre parties de la chaine. Ils ne sont pas
**prefixes** pour autant : 5 des 21 etats seulement sont des prefixes du mot de Thue-Morse, 6
pour Fibonacci ; les autres demarrent plus loin dans le mot. Le systeme de Post, lui, supprime
**trois** elements par pas : ce que la tete supprime depend de ce que la queue a recu, la
boucle causale est fermee, et la trace y laisse des trous -- `1111` est absent de la trace de
la graine de reference, et aucun bloc ajoute (`00` ou `1101`) ne peut le fabriquer, faute de
quatre `1` consecutifs ou de deux `1` terminaux. Une graine arbitraire, elle, en produit : les
496 graines de longueur 4 a 8 sont 121 a en faire apparaitre, et c'est l'exercice 3 qui le
mesure. La frontiere entre le systeme de substitution (compris) et le tag (ouvert) passe la --
entre `r = 1` et `r = 2`, un seul element supprime de plus.

## 9. Reception critique : un siecle de combinateurs, lue depuis 2021

La these wolframienne -- Schonfinkel fournit "the first explicit example of a system capable of
what we now call universal computation" (*S Combinator Challenge*, section *Hiding in Plain
Sight for a Century?*) -- se confronte a une tradition que Wolfram lui-meme a patiemment
relevee : *A Bibliography of Combinators* (arXiv 2106.11729) recense **plus de 200 entrees**
sur un siecle, de Curry (a partir de 1927-1930) aux travaux modernes de term rewriting.

| Volet de la reception | Reference | Ce qu'elle oppose a la these |
|---|---|---|
| Systematisation | Curry, Hindley & Seldin, *Combinatory Logic* vol. II (1972) | le programme n'est pas une decouverte isolee de 1920 : c'est quarante ans de construction de Curry, achevee en 1972 |
| Synthese moderne | Hindley & Seldin, *Lambda-Calculus and Combinators: an Introduction* (1980/2008) | le lambda-calcul et les combinateurs forment un seul corpus enseignable, pas un curiosity isole |
| Histoire | Cardone & Hindley, *History of Lambda-calculus and Combinatory Logic* (2006/2009) | les combinateurs se comprennent dans la continuite Frege-Principia-Church, pas comme un big bang |
| Acces canonique | Bimbo, *Combinatory Logic*, Stanford Encyclopedia of Philosophy (2009) | la communaute logique a son article de reference -- la place des combinateurs est institutionnalisee |
| Comportement computationnel | Klop, van Oostrom & van Raamsdonk, *Combinatory term rewriting systems* (TCS 121, 1993) | la theorie du rewriting a traite les combinateurs comme objet d'etude a part entiere, bien avant la ruliologie |

Ces references sont relevees dans la **bibliographie archivee de Wolfram lui-meme** (PDF
arXiv 2106.11729) -- le carnet cite la grille de lecture wolframienne et ses propres sources,
pas les originaux : c'est une reception de reception, et il le dit.

Le point de friction le plus net : la these de la **representation ultime**. Wolfram ecrit les
combinateurs comme le fondement le plus simple du calcul ; la tradition Curry-Hindley les traite
comme un **formalisme commode** -- une notation sans variables pour raisonner sur les fonctions
et les modeles de la logique combinatoire. Les deux lectures ne se contredisent pas ; elles ne
parlent pas de la meme question. Le S Combinator Challenge ($20 000, 2021) est la seule piece
qui les force a se rejoindre : si S seul est universel, la these du minimum absolu se renforce
de l'interieur du formalisme -- et le challenge est, a ce jour, non resolu.

## 10. Synthese -- trois statuts, un seul instrument

| Affirmation | Source | Statut | Ce que ce carnet a fait |
|---|---|---|---|
| S K K x -> x : I est derivable | Schonfinkel 1924 | theoreme trivial | reduit pas a pas (section 1) |
| L'ordre de reduction change l'issu : K I OMEGA | standardisation | theoreme | les deux ordres executes (section 2) |
| S/K represente l'arithmetique | Church 1936 | theoreme | 2+3, 2*3, 2^3 calcules (section 5) |
| Toute forme normale est unique | Church-Rosser | theoreme | **temoin empirique** : 30 ordres -> 1 forme (section 6) ; preuve formelle = grain B (Lean) |
| 1111 est inatteignable dans le tag de Post | Wolfram 2021, *The Basic Setup* | fait prouvable sur les regles | mesure sur la trace (section 8) |
| Le tag de Post s'arrete-t-il toujours ? | Post 1921 | **probleme ouvert** | comportements mesures, aucun tranche (section 7) |
| S seul est-il universel ? | Challenge 2021 | probleme ouvert ($20 000) | cadre expose, non attaque (section 9) |

Le pli 4 du Origami Wolfram demandait la reception critique "au meme rang que les theses" :
c'est la table ci-dessus -- chaque these a son statut, chaque statut a sa charge de preuve, et
le carnet ne depasse jamais la sienne.

## Vérifiez votre compréhension

1. La section 2 exécute les deux ordres de réduction sur `K I OMEGA` — quel ordre termine, et que montre celui qui ne termine pas ?
2. L'arithmétique de Church est calculée en section 5 — pourquoi `2^3` n'est-il pas symétrique à `3^2` ?
3. Church-Rosser est attesté en section 6 par 30 ordres vers une forme normale — pourquoi cela reste-t-il un témoin empirique et non une preuve ?

## Exercice 1 -- Le combinateur d'echange, en pur S/K

Le combinateur `C` echange ses deux derniers arguments : `C f x y -> f y x`. Construisez-le
par abstraction de bracket (`Combinateurs.Lam`, trois variables liees), appliquez-le a `f x y` et
verifiez que la forme normale est `f y (x)` -- c'est-a-dire `f y x` en association a gauche.

**Etapes** : (1) ecrire `C` comme `\f x y. f y x` via trois `Combinateurs.Lam` imbriques ; (2) construire
`C f x y` avec `new App(...)` ; (3) reduire en ordre normal et afficher la forme normale ; (4) compter
les noeuds de `C` avec la fonction `taille` de la section 4 -- le prix de l'echange.

In [10]:
#nullable enable
// Exercice 1 — Le combinateur d'echange, en pur S/K.
// TODO etudiant : construire C = \f x y. f y x par abstraction de bracket
// Etape 1 : utiliser successivement Combinateurs.Lam pour lier f, x et y.
// Etape 2 : appliquer C a f, x, y
// TODO etudiant
// Etapes 3-4 : reduire et mesurer
Term? C = null;         // TODO etudiant : le combinateur d'echange, en S/K pur
Term? applique = null;  // TODO etudiant : C applique a f, x, y

// Etapes 3-4 : le nombre de noeuds du terme, a la maniere de la section 4.
int TailleNoeuds(Term t)
{
    // TODO etudiant : nombre de noeuds (Var ou App) du terme.
    return 0;
}

if (C is null || applique is null)
{
    Console.WriteLine("Exercice a completer : C et applique");
}
else
{
    (Term? nf, int steps, List<Term>? _) = Combinateurs.ReduceTerm(applique!, "normal", 500);
    Console.WriteLine($"forme normale : {(nf is null ? "aucune" : Combinateurs.ToStr(nf))} ({steps} pas)");
    Console.WriteLine($"noeuds de C : {TailleNoeuds(C!)}");
}

Exercice a completer : C et applique


## Exercice 2 -- L'exponentiation n'est pas symetrique

La section 5 calcule `2^3 = 8` avec `exp = \m n. n m`. Verifiez que **l'ordre des operandes
compte** : `3^2` doit donner `9`, pas `8`. Construisez les deux termes, reduisez-les et
convertissez avec `Combinateurs.ChurchToInt`.

**Etapes** : (1) reconstruire `exp` par `Combinateurs.Lam` ; (2) `exp trois deux` et `exp deux trois` ;
(3) convertir les deux resultats ; (4) expliquer en une phrase pourquoi `n m` donne `m^n`
et non `m^n` inverse -- que represente l'application de `n` a `m` ?

In [11]:
#nullable enable
// Exercice 2 — L'exponentiation n'est pas symetrique.
// TODO etudiant : construire \m n. n m par Combinateurs.Lam.
// 3^2 attendu : 9
// 2^3 attendu : 8
Term deux = Combinateurs.Church(2);
Term trois = Combinateurs.Church(3);
Term? exp = null;   // TODO etudiant : \m n. n m par Combinateurs.Lam

if (exp is null)
{
    Console.WriteLine("Exercice a completer : exp");
}
else
{
    Term troisAuDeux = new App(new App(exp!, trois), deux);   // 3^2 attendu : 9
    Term deuxAuTrois = new App(new App(exp!, deux), trois);   // 2^3 attendu : 8
    Console.WriteLine($"3 ^ 2 = {Combinateurs.ChurchToInt(troisAuDeux)} | 2 ^ 3 = {Combinateurs.ChurchToInt(deuxAuTrois)}");
}

Exercice a completer : exp


## Exercice 3 -- Cartographier les graines du tag de Post

La section 7 teste quatre graines. Ecrivez un balayage : pour toutes les graines binaires de
longueur 4 a 8, executez `Combinateurs.TagEvolve` (borne `maxSteps=100000`), comptez les verdicts
`halted` / `cycle` / `unbounded`, et relevez la plus longue croissance avant cycle.

**Etapes** : (1) generer les graines avec le produit cartesien de `Enumerable.Repeat("01", L)` ;
(2) classer chaque verdict ; (3) afficher le decompte par longueur ; (4) relever la graine dont le
transitoire est le plus long -- la reponse reste nettement sous 100 etats, verifiez-le ;
(5) montrer que le transitoire peut pourtant depasser 100 etats, mais hors de ce domaine :
cherchez une graine de longueur superieure a 8 (il en existe une des la longueur 12), et
verifiez que votre mesure est reproductible (le systeme est deterministe : deux executions
donnent le meme verdict).

In [12]:
#nullable enable
// Exercice 3 — Cartographier les graines du tag de Post.
// TODO etudiant : {(L, verdict): nombre}
Dictionary<(int L, string Verdict), int>? decompte = null;   // TODO etudiant : {(L, verdict): nombre}
// TODO etudiant : (graine, nombre d'etats avant cycle)
(string Graine, int Etats)? transitoireMax = null;
// TODO etudiant : nombre total d'etats enregistres, cycle compris
int? traceMax = null;

if (decompte is null || transitoireMax is null || traceMax is null)
{
    Console.WriteLine("Exercice a completer : cartographier les graines de longueur 4 a 8");
    Console.WriteLine("Indice : Combinateurs.TagEvolve(graine) renvoie (trace, verdict). Distinguer trace.Count, qui inclut le cycle et l'etat repete, du transitoire avant le premier etat deja visite.");
}
else
{
    // TODO etudiant : le decompte par (longueur, verdict) et le transitoire maximal.
    foreach (var (cle, nombre) in decompte!.OrderBy(k => k.Key.L).ThenBy(k => k.Key.Verdict, StringComparer.Ordinal))
        Console.WriteLine($"longueur {cle.L} : verdict {cle.Verdict} -> {nombre} graines");

    Console.WriteLine($"transitoire maximal : graine {transitoireMax!.Value.Graine} | etats avant cycle : {transitoireMax.Value.Etats}");
    Console.WriteLine($"trace maximale : {traceMax} etats enregistres, cycle et etat repete inclus");
}

Exercice a completer : cartographier les graines de longueur 4 a 8


Indice : Combinateurs.TagEvolve(graine) renvoie (trace, verdict). Distinguer trace.Count, qui inclut le cycle et l'etat repete, du transitoire avant le premier etat deja visite.


## À retenir

*Carnet d'origine ICT (pli 4 de l'Origami Wolfram, anciennement ICT-51), rattaché à la série Complexity — voir le [registre des renommages](../../../docs/reference/rename-ledger.tsv).*

Ce carnet couvre les combinateurs S/K/I comme représentation du calcul — syntaxe, réduction, stratégies, arithmétique de Church — et la borne d'irréductibilité que leur oppose le système de tag de Post. Trois statuts coexistent sans se confondre : des théorèmes illustrés par des calculs, un témoin empirique de Church-Rosser et deux problèmes ouverts : l'arrêt du tag de Post et l'universalité de S seul.

La preuve formelle de la confluence et la lecture des textes de 2025 (*Ruliology of Lambdas*) prolongent ce parcours. Un moteur minimal n'est pas un moteur pauvre : S et K tiennent en deux lignes, et un siècle de réception n'a pas épuisé ce qu'elles contiennent.

## Pour aller plus loin

- **[Complexity-08c — Machines de Gödel](Complexity-08c-MachinesDeGoedel-Python.ipynb)** : le carnet suivant, où la self-modification prouvée prolonge la question de l'irréductibilité.
- **Jumeau Python** → [Complexity-08b-CombinateursSKI-TagDePost-Python](Complexity-08b-CombinateursSKI-TagDePost-Python.ipynb) — le carnet d'origine, en Python.
- **[ICT-Synthèse — CrossSubstrat](../../IIT/ICT-Series/ICT-Synthese-CrossSubstrat-Python.ipynb)** : le capstone de la série d'origine, qui mesure ce que les instruments transfèrent d'un substrat à l'autre.
- **[ICT-0 — Cadrage de la série](../../IIT/ICT-Series/ICT-0-Framing.md)** : le principe directeur de la série ICT d'origine.

## Attestation de parité

Les cellules de calcul rejouent les instances Python et contrôlent leurs résultats contre des références figées ; une différence déclenche une exception. Le banc de la section 6 complète ces contrôles par des comparaisons et des certificats, affiche leurs verdicts et termine par le nombre d'écarts. Les exercices des sections 10 restent indépendants de ces calculs.

| Section | Mesures comparées |
|---|---|
| 1 — Syntaxe | Trace exécutée de `S K K x`, forme normale et nombre de pas ; certificat complémentaire sur les transitions. |
| 2 — Stratégies | Deux réductions de `K I OMEGA` à la borne de 500 pas ; certificats complémentaires sur l'effacement et le cycle. |
| 3 — OMEGA | Terme initial, trace à trois pas et absence de forme normale observée à 200 pas. |
| 4 — Abstraction | Trois traductions produites par `Lam` et tailles comptées de `church(1)` à `church(5)` ; certificats complémentaires sur les traductions de référence. |
| 5 — Entiers de Church | Écriture de `church(2)`, aller-retour de 0 à 7 et opérations construites par abstraction sur 2 et 3. |
| 6 — Church-Rosser | Forme normale, décompte et statistiques de pas sur 30 tirages. Les générateurs Python et .NET ne sont pas supposés choisir les mêmes chemins. |
| 7 — Tag de Post | Quatre appels directs à `TagEvolve`, verdicts, tailles et extraits des traces aux bornes de la source ; certificats complémentaires par pas élémentaires. |
| 8 — Blocs et substitution | Blocs absents, mots de substitution et simulation des états à `r = 1`, avec leurs décomptes de préfixes et de facteurs. |

Les comparaisons déterministes sont exactes, sans tolérance : elles portent sur des entiers, des formes symboliques et des chaînes. Le banc ne dépend pas des réponses aux exercices.